# RepWise MCP lab
Build and connect an MCP server for workout plan generation.

This adapts the Session 3 campus exercise to RepWise. A real local Ollama agent discovers the tool through MCP, sends a workout request, receives a Pydantic result, and summarizes the saved plan.

Run locally from the repository with Python 3.11 or later. Install `requirements-notebook.txt`, prepare the corpus chunks, and start Ollama with `qwen3.5:4b` installed. No Colab, GPU setup or hosted API key is needed.

## 1 Set up the local environment
Select the project's Python environment as the notebook kernel. Run once in a terminal if needed:

```powershell
python -m pip install -r requirements-notebook.txt
ollama pull qwen3.5:4b
```

The exercise uses BM25 over the real corpus so it does not need a semantic index. Both the agent and plan generator use the model selected below. The normal application settings remain unchanged.

In [1]:
import asyncio
import json
import os
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the RepWise repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11 or newer kernel")

import httpx
from mcp import ClientSession
from mcp.client.stdio import stdio_client
from config.config import AppConfig
from src.agent.lab import local_lab
from src.agent.mcp_client import logged_stdio, run_workout_agent
from src.tools.generate_workout_plan import GenerateWorkoutPlanResult

MODEL = os.getenv("REPWISE_AGENT_MODEL", "qwen3.5:4b")
settings = AppConfig.from_environment()
with httpx.Client(base_url=settings.ollama_url, timeout=10) as client:
    response = client.get("/api/tags")
    response.raise_for_status()
    installed = [item["name"] for item in response.json()["models"]]
assert MODEL in installed, f"Run ollama pull {MODEL} first"
print("Python:", sys.version.split()[0])
print("Agent and generation model:", MODEL)
print("Retrieval: BM25, real corpus; scope screening enabled")

Python: 3.12.10
Agent and generation model: qwen3.5:4b
Retrieval: BM25, real corpus; scope screening enabled


## 2 The MCP server
The reusable server is in `src/mcp_server/server.py`. It exposes `generate_workout_plan` using the official MCP Python SDK's FastMCP class and stdio transport.

The model supplies the focus, workout count and optional constraints. The host supplies the signed-in account and token outside the tool arguments. The server calls the regular authenticated API, which runs the existing RAG pipeline and saves a validated plan. There is no separate model implementation in the MCP server.

The next function connects, lists the actual tool schema, then runs the agent. It does not call the generation tool by hand.

In [2]:
async def discover_and_run(goal, server_parameters):
    async with logged_stdio(server_parameters) as (read, write):
        async with ClientSession(read, write) as session:
            handshake = await session.initialize()
            tools = (await session.list_tools()).tools
            print("MCP server:", handshake.serverInfo.name)
            print("Discovered tools:", [tool.name for tool in tools])
            assert [tool.name for tool in tools] == ["generate_workout_plan"]
            assert tools[0].outputSchema is not None
            print("Input schema:")
            print(json.dumps(tools[0].inputSchema, indent=2))
            print("Structured output schema present:", bool(tools[0].outputSchema))
    return await run_workout_agent(
        goal,
        server_parameters,
        model=MODEL,
        ollama_url=settings.ollama_url,
        max_steps=5,
        timeout_seconds=240,
    )


print("MCP discovery and agent function ready")

MCP discovery and agent function ready


## 3 Give the agent one request
The lab starts a temporary backend on a free localhost port, registers a temporary account and saves a beginner profile: Monday, 45 minutes, bodyweight equipment. Login tokens are kept in memory and never displayed.

The agent chooses the tool arguments from the request below. The backend performs real retrieval and Ollama generation, including normal validation and correction retries. One plan-generation call is permitted to avoid duplicate saves.

In [3]:
GOAL = (
    "Generate and save a one-week beginner program to build leg muscles. "
    "I can train once, on Monday, for up to 45 minutes, using bodyweight only. "
    "Use my saved profile and the fitness evidence available in RepWise. "
    "Then briefly summarize the saved program."
)
print(GOAL)

Generate and save a one-week beginner program to build leg muscles. I can train once, on Monday, for up to 45 minutes, using bodyweight only. Use my saved profile and the fitness evidence available in RepWise. Then briefly summarize the saved program.


In [4]:
def run_lab_request(goal):
    # A worker event loop supports Windows subprocess pipes while Jupyter stays responsive.
    with local_lab(MODEL) as lab:
        print("Temporary account and profile ready; normal user database unchanged.")
        loop_factory = asyncio.ProactorEventLoop if os.name == "nt" else None
        with asyncio.Runner(loop_factory=loop_factory) as runner:
            outcome = runner.run(discover_and_run(goal, lab.server_parameters()))
        result = GenerateWorkoutPlanResult.model_validate(outcome["result"])
        assert result.status == "completed" and result.plan_id and result.plan
        assert len(outcome["tool_calls"]) == 1
        assert result.sources
        with httpx.Client(
            base_url=lab.api_url,
            timeout=10,
            headers={"Authorization": f"Bearer {lab.access_token}"},
        ) as api:
            response = api.get(f"/api/v1/students/{lab.user_id}/plans")
            response.raise_for_status()
            history = response.json()
        assert len(history) == 1, "Expected exactly one saved plan"
        assert history[0]["id"] == result.plan_id
        assert history[0]["plan"] == result.plan.model_dump(mode="json")
        assert history[0]["sources"] == [
            s.model_dump(mode="json") for s in result.sources
        ]
        outcome["verification"] = {
            "mcp_tool_calls": len(outcome["tool_calls"]),
            "status": result.status,
            "saved_plan_verified": True,
            "source_count": len(result.sources),
            "temporary_account": True,
        }
        return outcome


outcome = await asyncio.to_thread(run_lab_request, GOAL)
print("Temporary backend stopped and test database removed.")

Temporary account and profile ready; normal user database unchanged.


MCP server: RepWise
Discovered tools: ['generate_workout_plan']
Input schema:
{
  "properties": {
    "focus": {
      "maxLength": 300,
      "minLength": 3,
      "title": "Focus",
      "type": "string"
    },
    "workouts_per_week": {
      "anyOf": [
        {
          "maximum": 7,
          "minimum": 1,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Workouts Per Week"
    },
    "extra_constraints": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "maxItems": 10,
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Extra Constraints"
    },
    "refinement_instructions": {
      "anyOf": [
        {
          "maxLength": 500,
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Refinemen

MCP connected. Discovered tools: ['generate_workout_plan']


Agent tool call: generate_workout_plan {"focus": "build leg muscles", "workouts_per_week": 1, "extra_constraints": []}


MCP result: completed plan_id: b687abc6-8e38-447e-9862-6dfc79dc2406


Temporary backend stopped and test database removed.


## 4 Inspect the result
These outputs are from the real agent and server execution. A successful run must include an actual tool call, a completed Pydantic result and a matching saved database record. The temporary database is removed after verification; the returned JSON and execution trace remain in this notebook.

In [5]:
print("Agent trace:")
print(json.dumps(outcome["tool_calls"], indent=2, ensure_ascii=False))
print("\nAgent answer:")
print(outcome["answer"])
print("\nVerification:")
print(json.dumps(outcome["verification"], indent=2))
assert outcome["verification"]["saved_plan_verified"]
print(
    "\nSUCCESS: agent -> MCP server -> authenticated API -> RAG pipeline -> saved plan"
)

Agent trace:
[
  {
    "name": "generate_workout_plan",
    "arguments": {
      "focus": "build leg muscles",
      "workouts_per_week": 1,
      "extra_constraints": []
    },
    "status": "completed",
    "plan_id": "b687abc6-8e38-447e-9862-6dfc79dc2406"
  }
]

Agent answer:
A beginner leg-focused workout has been created and saved for you. The plan, titled "Beginner Leg Hypertrophy Plan," is designed to build muscle using only bodyweight exercises on Monday. It includes four exercises: Bodyweight Squats, Bodyweight Lunges, Glute Bridges, and Calf Raises, with a total estimated duration of 40 minutes. The program emphasizes high-repetition volume and proper form to stimulate hypertrophy without heavy equipment.

Verification:
{
  "mcp_tool_calls": 1,
  "status": "completed",
  "saved_plan_verified": true,
  "source_count": 5,
  "temporary_account": true
}

SUCCESS: agent -> MCP server -> authenticated API -> RAG pipeline -> saved plan


In [6]:
result = GenerateWorkoutPlanResult.model_validate(outcome["result"])
print(result.model_dump_json(indent=2))

{
  "status": "completed",
  "message": "Plan created from retrieved sources.",
  "plan": {
    "title": "Beginner Leg Hypertrophy Plan",
    "summary": "A single leg-focused workout designed for a beginner to build muscle using only bodyweight. The plan emphasizes high-repetition volume and time-under-tension to stimulate hypertrophy without heavy equipment.",
    "weeks": [
      {
        "number": 1,
        "objective": "Establish movement patterns and build muscular endurance in the legs using bodyweight exercises.",
        "workouts": [
          {
            "day": "Monday",
            "focus": "Leg Hypertrophy",
            "estimated_minutes": 40,
            "exercises": [
              {
                "name": "Bodyweight Squat",
                "sets": 3,
                "repetitions": "15-20",
                "rest_seconds": 60,
                "estimated_minutes": 8,
                "coaching_note": "Focus on depth and control. Do not go to failure; stop when form br

## What this demonstrates
- Tool discovery and a real stdio MCP request chosen by an Ollama agent.
- An authenticated backend call without exposing credentials to the model.
- Real corpus retrieval, generated workouts, Pydantic output and persistence verification.
- A bounded loop that raises on tool failures, missing calls or duplicate generation.

This notebook does not implement nearby-place search or scheduling. To connect an external MCP host to an existing account, see `docs/agent/mcp_lab.md`. The API must be running, and the host must privately provide the API URL, user ID and access token.

Protocol reference: https://py.sdk.modelcontextprotocol.io/v1/